# Week 5: Local Search: Simulated Annealing for 8-Queens

The Week 5 slides name the 8-Queens problem. The accompanying PDF provides the representation, cost function, neighbor rule, acceptance probability and cooling schedule used here.

**Source:** `PPTs/Week5.pptx`.


## Program — Simulated annealing for N-Queens

**Theory.** Simulated annealing can accept a worse move with probability exp(-ΔE/T), allowing escape from local minima while temperature is high. The supplied Week 5 PDF explains a one-queen-per-column board representation, pair-conflict cost, random row changes and geometric cooling. The PPT specifies the 8-Queens problem.

**Objective.** Find a zero-conflict board using user-set size, initial state, temperature, cooling rate and stopping temperature.

**Libraries.** Python standard library (`math`, `random`).

**Algorithm.**
1. Count attacking queen pairs. 2. Choose a random column and a different row. 3. Always accept an improvement; otherwise accept with probability e^(-ΔE/T). 4. Multiply T by the cooling factor. 5. Stop at cost zero, minimum temperature or iteration cap.

**Complexity.** O(k n²) time for k iterations and O(n) space.

The code below keeps `input()` so you can provide your own values. The final lines run a sample interaction through the same prompts; edit `SAMPLE_INPUTS` to try another case, or call the main function directly for a live terminal run.

**Expected output and explanation.** The sample fixes the random seed for a repeatable run; the program accepts any valid n-queen state and stops safely at the chosen iteration cap.


In [1]:
from contextlib import contextmanager
import builtins

@contextmanager
def sample_input(values):
    original_input = builtins.input
    remaining = iter(map(str, values))
    def demo_input(prompt=''):
        print(prompt, end='')
        try:
            value = next(remaining)
        except StopIteration as exc:
            raise RuntimeError('Add more values to SAMPLE_INPUTS for this run.') from exc
        print(value)
        return value
    builtins.input = demo_input
    try:
        yield
    finally:
        builtins.input = original_input

import math, random
def cost(board):
    return sum(board[i]==board[j] or abs(board[i]-board[j])==j-i
               for i in range(len(board)) for j in range(i+1,len(board)))
def simulated_annealing(board,temp,alpha,min_temp,max_steps,seed=None):
    rng=random.Random(seed); board=board[:]; best=board[:]; step=0
    while temp>min_temp and cost(board)>0 and step<max_steps:
        col=rng.randrange(len(board)); row=rng.choice([r for r in range(len(board)) if r!=board[col]])
        neighbor=board[:]; neighbor[col]=row
        delta=cost(neighbor)-cost(board)
        if delta<0 or rng.random()<math.exp(-delta/temp): board=neighbor
        if cost(board)<cost(best): best=board[:]
        step+=1
        if step%20==0 or cost(board)==0:
            print(f'Step {step}: cost={cost(board)}, temperature={temp:.4f}')
        temp*=alpha
    print(f'Iterations: {step}')
    return best
def main():
    try:
        n=int(input('Number of queens (8 for the slide problem): '))
        board=list(map(int,input('Enter n row numbers (0 through n-1): ').split()))
        temp=float(input('Initial temperature (>0): ')); alpha=float(input('Cooling factor (0<alpha<1): '))
        min_temp=float(input('Minimum temperature (>0): ')); max_steps=int(input('Maximum iterations: '))
    except ValueError: print('Please enter numeric values.'); return
    if n<2 or len(board)!=n or any(r<0 or r>=n for r in board) or temp<=0 or not 0<alpha<1 or min_temp<=0 or max_steps<1:
        print('Invalid input.'); return
    print(f'Initial cost: {cost(board)}')
    result=simulated_annealing(board,temp,alpha,min_temp,max_steps,seed=23)
    print('Final board:',result); print('Final cost:',cost(result))
    print('Solution found!' if cost(result)==0 else 'Stopped before finding a solution.')


SAMPLE_INPUTS = ['8', '0 1 2 3 4 5 6 7', '10', '0.95', '0.01', '1500']
with sample_input(SAMPLE_INPUTS):
    main()


Number of queens (8 for the slide problem): 8
Enter n row numbers (0 through n-1): 0 1 2 3 4 5 6 7
Initial temperature (>0): 10
Cooling factor (0<alpha<1): 0.95
Minimum temperature (>0): 0.01
Maximum iterations: 1500
Initial cost: 28
Step 20: cost=7, temperature=3.7735
Step 40: cost=5, temperature=1.3528
Step 60: cost=5, temperature=0.4849
Step 80: cost=4, temperature=0.1738
Step 100: cost=3, temperature=0.0623
Step 120: cost=2, temperature=0.0223
Step 130: cost=0, temperature=0.0134
Iterations: 130
Final board: [4, 6, 1, 5, 2, 0, 3, 7]
Final cost: 0
Solution found!
